# Stage 2: how well does it perform on real data

Two datasets with published ground truth: a MERFISH small intestine section and APEX-seq HEK293.
Each method scores every gene by how far its real distribution sits from the same set of randomly
repositioned transcripts, and calls it localised at a cut-off derived per method on simulated
data — different classifiers put the distance on different scales.

In [1]:
import pandas as pd

import config
import loading
import scoring
import tables

%load_ext autoreload
%autoreload 2

## Small intestine

In [2]:
ileum_real, ileum_control = loading.real_pair("ileum")
f"{ileum_real.n_obs} gene-cell pairs over {ileum_real.obs['original_obs_index'].nunique()} cells, {ileum_real.obs['gene_name'].nunique()} genes tested"

/mnt/data/miniconda3/envs/subcellspace_env/lib/python3.9/site-packages/anndata/_core/aligned_mapping.py:54: ExperimentalFeatureWarning: Support for Awkward Arrays is currently experimental. Behavior may change in the future. Please report any issues you may encounter!
  warnings.warn(


'3273 gene-cell pairs over 406 cells, 30 genes tested'

In [3]:
ileum_rows = scoring.stage2("ileum")
ileum_rows.shape

(340, 22)

## APEX-seq

In [4]:
apex_real, apex_control = loading.real_pair("apex_hek")
# only genes with an APEX-seq label (localised or control) are scored
labelled = loading.ground_truth("apex_hek", [str(g) for g in apex_real.obs["gene_name"].unique()])
f"{apex_real.n_obs} gene-cell pairs over {apex_real.obs['original_obs_index'].nunique()} cells, {len(labelled.positives | labelled.negatives)} genes with an APEX-seq label tested"

'8449041 gene-cell pairs over 150151 cells, 250 genes with an APEX-seq label tested'

In [5]:
apex_rows = scoring.stage2("apex_hek")
apex_rows.shape

(807, 22)

## Save

In [6]:
scoring.save([ileum_rows, apex_rows], "stage2")
results = scoring.combine()
results.shape

(18319, 14)

## Results

In [8]:
for dataset, negatives, positives in [("ileum", 13, 17), ("apex_hek", 26, 224)]:
    calls = pd.concat([
        tables.mean_per_method(results, dataset, "detection_at_threshold", [m]).rename(index={m: name})
        for m, name in [("tp", f"genes found (of {positives})"), ("fp", f"false calls (of {negatives})"),
                        ("specificity", "specificity"), ("precision", "precision"), ("mcc", "MCC")]
    ])
    calls.index.name = dataset
    tables.show(tables.label_columns(calls), f"{dataset}: calls at each method's cut-off")

**ileum: calls at each method's cut-off**

| ileum               | SubCellSpace * | Point-VAE | CNN baseline * | Bento  | ELLA    | Image transformer (small) * | Image transformer (large) * |
|---------------------|---------------:|----------:|---------------:|-------:|--------:|----------------------------:|----------------------------:|
| genes found (of 17) |             13 |         4 |             12 |     14 |      16 |                          12 |                          11 |
| false calls (of 13) |              1 |         1 |              0 |      3 |      13 |                           1 |                           1 |
| specificity         |         0.9231 |    0.9231 |         1.0000 | 0.7692 |  0.0000 |                      0.9231 |                      0.9231 |
| precision           |         0.9286 |    0.8000 |         1.0000 | 0.8235 |  0.5517 |                      0.9231 |                      0.9167 |
| MCC                 |         0.6832 |    0.2106 |         0.7140 | 0.5928 | -0.1624 |                      0.6290 |                      0.5767 |

`*` encoder variations on the SubCellSpace pipeline.

**apex_hek: calls at each method's cut-off**

| apex_hek             | SubCellSpace * | Point-VAE | CNN baseline * | Bento  | ELLA   | Image transformer (small) * | Image transformer (large) * |
|----------------------|---------------:|----------:|---------------:|-------:|-------:|----------------------------:|----------------------------:|
| genes found (of 224) |             69 |        35 |            152 |    196 |    224 |                         190 |                         178 |
| false calls (of 26)  |              1 |         1 |             10 |     21 |     26 |                          21 |                          16 |
| specificity          |         0.9615 |    0.9615 |         0.6154 | 0.1923 | 0.0000 |                      0.1923 |                      0.3846 |
| precision            |         0.9857 |    0.9722 |         0.9383 | 0.9032 | 0.8960 |                      0.9005 |                      0.9175 |
| MCC                  |         0.1833 |    0.1024 |         0.1879 | 0.0607 | 0.0000 |                      0.0341 |                      0.1312 |

`*` encoder variations on the SubCellSpace pipeline.

In [9]:
subtype = pd.concat([
    tables.mean_per_method(results, "apex_hek", "subtype_agreement", [m]).rename(index={m: name})
    for m, name in [("n_genes", "genes assigned"), ("adjusted_rand", "adjusted Rand"),
                    ("adjusted_mutual_info", "adjusted mutual information")]
])
subtype.index.name = "APEX-seq"
tables.show(tables.label_columns(subtype), "Pattern subtypes against APEX compartments")

**Pattern subtypes against APEX compartments**

| APEX-seq                    | SubCellSpace * | Point-VAE | CNN baseline * | Bento  | ELLA   | Image transformer (small) * | Image transformer (large) * |
|-----------------------------|---------------:|----------:|---------------:|-------:|-------:|----------------------------:|----------------------------:|
| genes assigned              |             69 |        35 |            152 |    196 |    224 |                         190 |                         178 |
| adjusted Rand               |         0.1238 |    0.0481 |         0.0606 | 0.0601 | 0.1268 |                      0.0476 |                      0.0791 |
| adjusted mutual information |         0.1912 |    0.1800 |         0.1114 | 0.1108 | 0.1862 |                      0.1229 |                      0.1554 |

`*` encoder variations on the SubCellSpace pipeline.